# Evaluacion en TEST: RetinexNet base vs RetinexNet + WGAN-GP

Notebook standalone que **no reentrena nada**: carga los pesos finales del modelo 1 (WGAN-GP) desde `dev/weights/2a_Retinex-GAN/` y los pesos base preentrenados de `aasharma90/RetinexNet_PyTorch`, reconstruye el split 80/10/10 (seed 42) usado en el resto del proyecto y evalua PSNR/SSIM de ambos modelos sobre el **test set**.

Resultado comparable con los modelos 2 (RetinexNet+DenoiseNet) y 3 (Zero-DCE-FT++), que tambien reportan sobre test.

In [1]:
#@title 1. Setup: entorno, PROJECT_ROOT y RetinexNet_PyTorch (pesos base)
import os, subprocess, sys, pathlib

IN_COLAB = "google.colab" in sys.modules or os.path.isdir("/content")

if IN_COLAB:
    PROJECT_ROOT = "/content/LowLight-Enhancement"
    if not os.path.isdir(PROJECT_ROOT):
        subprocess.check_call(["git", "clone", "--quiet", "https://github.com/jeanpaulmst/LowLight-Enhancement.git", PROJECT_ROOT], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
else:
    def _find_root():
        starts = [pathlib.Path(os.getcwd())]
        vsc_file = globals().get("__vsc_ipynb_file__")
        if vsc_file:
            starts.insert(0, pathlib.Path(vsc_file).parent)
        for start in starts:
            p = start.resolve()
            for _ in range(10):
                if (p / "CLAUDE.md").exists() or (p / "dev" / "models").is_dir():
                    return str(p)
                if p == p.parent:
                    break
                p = p.parent
        return None

    PROJECT_ROOT = _find_root()
    if PROJECT_ROOT is None:
        raise RuntimeError(
            "No se encontro el directorio raiz. "
            "Ajusta PROJECT_ROOT manualmente al inicio de esta celda."
        )

print("Entorno:", "Google Colab" if IN_COLAB else "Local")
print("PROJECT_ROOT:", PROJECT_ROOT)

# Repo base de RetinexNet (solo para los pesos preentrenados originales)
RETINEX_BASE_DIR = os.path.join(PROJECT_ROOT, "RetinexNet_PyTorch")
if not os.path.isdir(os.path.join(RETINEX_BASE_DIR, "ckpts")):
    subprocess.check_call(["git", "clone", "--quiet", "https://github.com/aasharma90/RetinexNet_PyTorch.git", RETINEX_BASE_DIR], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

WEIGHTS_DIR = os.path.join(PROJECT_ROOT, "dev", "weights", "2a_Retinex-GAN")
assert os.path.isdir(WEIGHTS_DIR), f"No se encontro la carpeta de pesos: {WEIGHTS_DIR}"
print("WEIGHTS_DIR:", WEIGHTS_DIR)
print("Pesos disponibles:", os.listdir(WEIGHTS_DIR))

Entorno: Google Colab
PROJECT_ROOT: /content/LowLight-Enhancement
WEIGHTS_DIR: /content/LowLight-Enhancement\dev\weights\2a_Retinex-GAN
Pesos disponibles: ['retinex_decom_net.pth', 'retinex_relight_net.pth']


In [ ]:
#@title 2. Descarga del dataset (solo si falta, idem notebooks anteriores)
LOLv1_DIR = os.path.join(PROJECT_ROOT, "data/raw/lol-dataset/lol_dataset/our485/low")
LOLv2_DIR = os.path.join(PROJECT_ROOT, "data/raw/lolv2-real/Train/Input")

need_v1 = not os.path.isdir(LOLv1_DIR)
need_v2 = not os.path.isdir(LOLv2_DIR)

if need_v1 or need_v2:
    import zipfile
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "kaggle", "huggingface_hub"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

    os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"  # silenciar barras de progreso de descarga

    if need_v1:
        os.environ["KAGGLE_API_TOKEN"] = "KGAT_266e07212debf3797d4b3d9e9786aabb"
        import kaggle
        kaggle.api.authenticate()

        dst = os.path.join(PROJECT_ROOT, "data/raw")
        os.makedirs(dst, exist_ok=True)
        kaggle.api.dataset_download_files("soumikrakshit/lol-dataset", path=dst, quiet=True)

        zip_path = os.path.join(dst, "lol-dataset.zip")
        with zipfile.ZipFile(zip_path) as z:
            z.extractall(os.path.join(dst, "lol-dataset"))
        os.remove(zip_path)
        print("LOL-v1 descargado.")

    if need_v2:
        from huggingface_hub import snapshot_download
        snapshot_download(
            repo_id="okhater/lolv2-real",
            repo_type="dataset",
            local_dir=os.path.join(PROJECT_ROOT, "data/raw/lolv2-real"),
            ignore_patterns=["*.gitignore"],
        )
        print("LOL-v2-real descargado.")
else:
    print("Dataset ya disponible en data/raw/.")

In [6]:
#@title 3. Arquitecturas DecomNet y RelightNet (identicas a 02a)
import torch
import torch.nn as nn
import torch.nn.functional as F


class DecomNet(nn.Module):
    def __init__(self, channel=64, kernel_size=3):
        super().__init__()
        self.net1_conv0 = nn.Conv2d(4, channel, kernel_size * 3, padding=4, padding_mode='replicate')
        self.net1_convs = nn.Sequential(
            nn.Conv2d(channel, channel, kernel_size, padding=1, padding_mode='replicate'), nn.ReLU(),
            nn.Conv2d(channel, channel, kernel_size, padding=1, padding_mode='replicate'), nn.ReLU(),
            nn.Conv2d(channel, channel, kernel_size, padding=1, padding_mode='replicate'), nn.ReLU(),
            nn.Conv2d(channel, channel, kernel_size, padding=1, padding_mode='replicate'), nn.ReLU(),
            nn.Conv2d(channel, channel, kernel_size, padding=1, padding_mode='replicate'), nn.ReLU(),
        )
        self.net1_conv_out = nn.Conv2d(channel, 4, kernel_size, padding=1, padding_mode='replicate')

    def forward(self, input_im):
        input_max = torch.max(input_im, dim=1, keepdim=True)[0]
        input_img = torch.cat((input_max, input_im), dim=1)
        feats     = self.net1_conv0(input_img)
        feats     = self.net1_convs(feats)
        out       = self.net1_conv_out(feats)
        R         = torch.sigmoid(out[:, 0:3, :, :])
        I         = torch.sigmoid(out[:, 3:4, :, :])
        return R, I


class RelightNet(nn.Module):
    def __init__(self, channel=64, kernel_size=3):
        super().__init__()
        self.net2_conv0_1   = nn.Conv2d(4,           channel,     kernel_size, padding=1, padding_mode='replicate')
        self.net2_conv1_1   = nn.Conv2d(channel,     channel,     kernel_size, stride=2, padding=1, padding_mode='replicate')
        self.net2_conv1_2   = nn.Conv2d(channel,     channel,     kernel_size, stride=2, padding=1, padding_mode='replicate')
        self.net2_conv1_3   = nn.Conv2d(channel,     channel,     kernel_size, stride=2, padding=1, padding_mode='replicate')
        self.net2_deconv1_1 = nn.Conv2d(channel * 2, channel,     kernel_size, padding=1, padding_mode='replicate')
        self.net2_deconv1_2 = nn.Conv2d(channel * 2, channel,     kernel_size, padding=1, padding_mode='replicate')
        self.net2_deconv1_3 = nn.Conv2d(channel * 2, channel,     kernel_size, padding=1, padding_mode='replicate')
        self.net2_fusion    = nn.Conv2d(channel * 3, channel,     kernel_size=1, padding=1)
        self.net2_output    = nn.Conv2d(channel,     1,           kernel_size,   padding=0)

    def forward(self, input_L, input_R):
        input_img  = torch.cat((input_R, input_L), dim=1)
        out0       = self.net2_conv0_1(input_img)
        out1       = F.relu(self.net2_conv1_1(out0))
        out2       = F.relu(self.net2_conv1_2(out1))
        out3       = F.relu(self.net2_conv1_3(out2))
        out3_up    = F.interpolate(out3,    size=(out2.shape[2], out2.shape[3]))
        deconv1    = F.relu(self.net2_deconv1_1(torch.cat((out3_up,  out2), dim=1)))
        deconv1_up = F.interpolate(deconv1, size=(out1.shape[2], out1.shape[3]))
        deconv2    = F.relu(self.net2_deconv1_2(torch.cat((deconv1_up, out1), dim=1)))
        deconv2_up = F.interpolate(deconv2, size=(out0.shape[2], out0.shape[3]))
        deconv3    = F.relu(self.net2_deconv1_3(torch.cat((deconv2_up, out0), dim=1)))
        deconv1_rs = F.interpolate(deconv1, size=(deconv3.shape[2], deconv3.shape[3]))
        deconv2_rs = F.interpolate(deconv2, size=(deconv3.shape[2], deconv3.shape[3]))
        fused      = self.net2_fusion(torch.cat((deconv1_rs, deconv2_rs, deconv3), dim=1))
        output     = self.net2_output(fused)
        return output


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print('DecomNet y RelightNet definidas.')

Device: cpu
DecomNet y RelightNet definidas.


In [7]:
#@title 4. Dataset LOLv1 + LOLv2-real: reconstruir split y test_loader (seed 42)
import random
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader, ConcatDataset, random_split

LOLv1_LOW  = os.path.join(PROJECT_ROOT, 'data/raw/lol-dataset/lol_dataset/our485/low')
LOLv1_HIGH = os.path.join(PROJECT_ROOT, 'data/raw/lol-dataset/lol_dataset/our485/high')
LOLv2_LOW  = os.path.join(PROJECT_ROOT, 'data/raw/lolv2-real/Train/Input')
LOLv2_HIGH = os.path.join(PROJECT_ROOT, 'data/raw/lolv2-real/Train/GT')

for p in [LOLv1_LOW, LOLv1_HIGH, LOLv2_LOW, LOLv2_HIGH]:
    assert os.path.isdir(p), f'Ruta no encontrada: {p}'
print('Rutas del dataset verificadas.')


class LowLightDataset(Dataset):
    def __init__(self, low_dir, high_dir):
        self.low_dir   = low_dir
        self.high_dir  = high_dir
        self.filenames = sorted(os.listdir(low_dir))

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):
        fname    = self.filenames[idx]
        low_img  = Image.open(os.path.join(self.low_dir,  fname)).convert('RGB')
        high_img = Image.open(os.path.join(self.high_dir, fname)).convert('RGB')
        low  = torch.from_numpy(np.array(low_img,  dtype=np.float32) / 255.0).permute(2, 0, 1)
        high = torch.from_numpy(np.array(high_img, dtype=np.float32) / 255.0).permute(2, 0, 1)
        return low, high


class AugmentSubset(Dataset):
    def __init__(self, subset, augment=False):
        self.subset  = subset
        self.augment = augment

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, idx):
        low, high = self.subset[idx]
        low  = Image.fromarray((low.permute(1, 2, 0).numpy()  * 255).astype(np.uint8))
        high = Image.fromarray((high.permute(1, 2, 0).numpy() * 255).astype(np.uint8))

        if self.augment:
            i = random.randint(0, low.height - 128)
            j = random.randint(0, low.width  - 128)
            low  = low.crop((j, i, j + 128, i + 128))
            high = high.crop((j, i, j + 128, i + 128))
            if random.random() > 0.5:
                low  = low.transpose(Image.FLIP_LEFT_RIGHT)
                high = high.transpose(Image.FLIP_LEFT_RIGHT)
            if random.random() > 0.5:
                low  = low.transpose(Image.FLIP_TOP_BOTTOM)
                high = high.transpose(Image.FLIP_TOP_BOTTOM)

        low  = torch.from_numpy(np.array(low,  dtype=np.float32) / 255.0).permute(2, 0, 1)
        high = torch.from_numpy(np.array(high, dtype=np.float32) / 255.0).permute(2, 0, 1)
        return low, high


dataset_v1 = LowLightDataset(LOLv1_LOW, LOLv1_HIGH)
dataset_v2 = LowLightDataset(LOLv2_LOW, LOLv2_HIGH)

full_dataset = ConcatDataset([dataset_v1, dataset_v2])
n       = len(full_dataset)
n_train = int(0.80 * n)
n_val   = int(0.10 * n)
n_test  = n - n_train - n_val

train_set, val_set, test_set = random_split(
    full_dataset,
    [n_train, n_val, n_test],
    generator=torch.Generator().manual_seed(42)
)

test_data   = AugmentSubset(test_set, augment=False)
test_loader = DataLoader(test_data, batch_size=8, shuffle=False, num_workers=0)

print(f'Total: {n} | Train: {n_train} | Val: {n_val} | Test: {len(test_data)}')

Rutas del dataset verificadas.
Total: 1174 | Train: 939 | Val: 117 | Test: 118


In [8]:
#@title 5. Cargar pesos: modelo base (preentrenado) y modelo GAN (fine-tuned)
import glob

def latest_ckpt(folder):
    files = sorted(
        glob.glob(f'{folder}/*.tar'),
        key=lambda x: int(os.path.splitext(os.path.basename(x))[0])
    )
    assert files, f'No hay checkpoints en {folder}'
    return files[-1]

# --- Modelo base (pesos originales de aasharma90/RetinexNet_PyTorch) ---
decom_base   = DecomNet().to(device)
relight_base = RelightNet().to(device)

ckpt_decom   = latest_ckpt(os.path.join(RETINEX_BASE_DIR, 'ckpts/Decom'))
ckpt_relight = latest_ckpt(os.path.join(RETINEX_BASE_DIR, 'ckpts/Relight'))

raw_decom = torch.load(ckpt_decom, map_location=device)
decom_state = {
    k.replace('net1_recon.', 'net1_conv_out.'): v
    for k, v in raw_decom.items()
}
decom_base.load_state_dict(decom_state)
relight_base.load_state_dict(torch.load(ckpt_relight, map_location=device))
decom_base.eval()
relight_base.eval()
print(f'Modelo base cargado desde {ckpt_decom} / {ckpt_relight}')

# --- Modelo GAN (pesos finales fine-tuned, dev/weights/2a_Retinex-GAN) ---
decom_gan   = DecomNet().to(device)
relight_gan = RelightNet().to(device)

decom_gan_path   = os.path.join(WEIGHTS_DIR, 'retinex_decom_net.pth')
relight_gan_path = os.path.join(WEIGHTS_DIR, 'retinex_relight_net.pth')

decom_gan.load_state_dict(torch.load(decom_gan_path, map_location=device))
relight_gan.load_state_dict(torch.load(relight_gan_path, map_location=device))
decom_gan.eval()
relight_gan.eval()
print(f'Modelo GAN cargado desde {decom_gan_path} / {relight_gan_path}')

Modelo base cargado desde /content/LowLight-Enhancement\RetinexNet_PyTorch\ckpts/Decom\9200.tar / /content/LowLight-Enhancement\RetinexNet_PyTorch\ckpts/Relight\9200.tar
Modelo GAN cargado desde /content/LowLight-Enhancement\dev\weights\2a_Retinex-GAN\retinex_decom_net.pth / /content/LowLight-Enhancement\dev\weights\2a_Retinex-GAN\retinex_relight_net.pth


In [9]:
#@title 6. Funcion de evaluacion (PSNR / SSIM)
from skimage.metrics import peak_signal_noise_ratio as psnr_fn
from skimage.metrics import structural_similarity as ssim_fn


@torch.no_grad()
def eval_model(decom, relight, loader):
    psnr_scores, ssim_scores = [], []
    for low, high in loader:
        low, high = low.to(device), high.to(device)
        R, I    = decom(low)
        I_delta = relight(I, R)
        h, w    = I_delta.shape[2], I_delta.shape[3]
        enhanced = torch.clamp(R[:, :, :h, :w] * I_delta, 0.0, 1.0)
        gt       = high[:, :, :h, :w]

        for i in range(enhanced.size(0)):
            pred_np = enhanced[i].cpu().numpy().transpose(1, 2, 0)
            gt_np   = gt[i].cpu().numpy().transpose(1, 2, 0)
            psnr_scores.append(psnr_fn(gt_np, pred_np, data_range=1.0))
            ssim_scores.append(ssim_fn(gt_np, pred_np, data_range=1.0, channel_axis=2))

    return float(np.mean(psnr_scores)), float(np.mean(ssim_scores))

In [10]:
#@title 7. Evaluacion sobre el TEST set
print('Evaluando modelo base sobre test...')
psnr_base_test, ssim_base_test = eval_model(decom_base, relight_base, test_loader)

print('Evaluando modelo GAN sobre test...')
psnr_gan_test, ssim_gan_test = eval_model(decom_gan, relight_gan, test_loader)

print()
print(f'{"Modelo":<12} {"PSNR (dB)":>12} {"SSIM":>10}')
print('-' * 36)
print(f'{"Base":<12} {psnr_base_test:>12.2f} {ssim_base_test:>10.4f}')
print(f'{"GAN":<12} {psnr_gan_test:>12.2f} {ssim_gan_test:>10.4f}')
print()
delta_psnr_test = psnr_gan_test - psnr_base_test
delta_ssim_test = ssim_gan_test - ssim_base_test
sign_p = '+' if delta_psnr_test >= 0 else ''
sign_s = '+' if delta_ssim_test >= 0 else ''
print(f'Delta:        {sign_p}{delta_psnr_test:.2f} dB  |  {sign_s}{delta_ssim_test:.4f}')

Evaluando modelo base sobre test...
Evaluando modelo GAN sobre test...

Modelo          PSNR (dB)       SSIM
------------------------------------
Base                15.81     0.5278
GAN                 17.12     0.6323

Delta:        +1.31 dB  |  +0.1045
